# Evaluación completa — zero-shot vs Qwen2.5-VL-3B + LoRA (data sintética)

**Antes de "Save & Run All":**
- Accelerator: **GPU T4 x2** · Internet: **ON**
- Inputs: `icarpediem/rodri-dataset`, notebook `andyjohan/entrenamiento-historias-clinicas` (trae `synth-lora.zip`) y notebook `andyjohan/data-sintetica` (trae `synth.zip`)

**Protocolo idéntico para todos:** misma muestra (seed 42), fp16, greedy, `max_new_tokens=128`, mismo prompt, misma normalización, batch 1, 3 imágenes de warm-up.
Si un modelo falla, se registra el error y el notebook **continúa** con el siguiente. Al final todo queda en `/kaggle/working/resultados_completos.zip`.

In [ ]:
# 0. Instalación (torchao viejo rompe peft → se desinstala ANTES de importar nada)
!pip uninstall -y -q torchao
!pip install -q rapidfuzz timm einops peft
import os, gc, re, glob, json, time, random, traceback, unicodedata, zipfile
import numpy as np, pandas as pd, torch, transformers
from pathlib import Path
from PIL import Image
from tqdm.auto import tqdm
from rapidfuzz.distance import Levenshtein
print("transformers:", transformers.__version__, "| torch:", torch.__version__)
print("GPUs:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

In [ ]:
# 1. Configuración
SEED, N_SAMPLES, MAX_NEW_TOKENS, N_WARMUP = 42, 500, 128, 3
DTYPE = torch.float16
PROMPT = ("Transcribe literalmente el texto manuscrito de esta imagen. "
          "Responde únicamente con la transcripción, sin comillas ni explicaciones.")

# Qué correr (en orden de prioridad: lo importante primero)
PLAN = [
    # (test,      modelo,        activo)
    ("rodrigo", "qwen25vl3b",      True),
    ("rodrigo", "qwen25vl3b_lora", True),
    ("synth",   "qwen25vl3b",      True),
    ("synth",   "qwen25vl3b_lora", True),
    ("rodrigo", "trocr",           True),
    ("rodrigo", "florence2",       True),
    ("rodrigo", "llava16",         True),
    ("rodrigo", "qwen25vl",        True),   # 7B, referencia
    ("synth",   "trocr",           True),   # zero-shot en dominio clínico (barato)
    ("synth",   "florence2",       True),
    ("synth",   "llava16",         False),  # ~2.5 h en 3077 líneas: activar solo si sobra tiempo
]

WORK = Path("/kaggle/working"); WORK.mkdir(exist_ok=True)
OUT = {"rodrigo": WORK / "bench_rodrigo", "synth": WORK / "bench_synth"}
for p in OUT.values(): p.mkdir(exist_ok=True)

def find_one(pattern, exclude=None):
    hits = sorted(glob.glob(pattern, recursive=True))
    if exclude: hits = [h for h in hits if exclude not in h.lower()]
    return hits[0] if hits else None

RODRIGO   = find_one("/kaggle/input/**/rodri-dataset")
SYNTH_ZIP = find_one("/kaggle/input/**/synth.zip")
LORA_ZIP  = find_one("/kaggle/input/**/synth-lora.zip")
print("Rodrigo:", RODRIGO, "\nsynth.zip:", SYNTH_ZIP, "\nsynth-lora.zip:", LORA_ZIP)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

In [ ]:
# 2. Utilidades: normalización, métricas, memoria
def norm_basic(s):
    s = unicodedata.normalize("NFC", str(s))
    return re.sub(r"\s+", " ", s).strip()

def norm_strict(s):
    s = norm_basic(s).lower()
    s = "".join(c for c in unicodedata.normalize("NFD", s) if unicodedata.category(c) != "Mn")
    s = re.sub(r"[^\w\s]", "", s)
    return re.sub(r"\s+", " ", s).strip()

def clean_output(s):
    s = norm_basic(s)
    s = re.sub(r"^(transcripci[oó]n|texto|text|transcription)\s*:\s*", "", s, flags=re.I)
    return s.strip('"“”\'«» ')

def cer_wer(refs, hyps):
    ce = sum(Levenshtein.distance(r, h) for r, h in zip(refs, hyps)); ct = sum(len(r) for r in refs)
    we = sum(Levenshtein.distance(r.split(), h.split()) for r, h in zip(refs, hyps)); wt = sum(len(r.split()) for r in refs)
    return 100 * ce / max(ct, 1), 100 * we / max(wt, 1)

def compute_metrics(preds):
    refs = preds["ref"].astype(str).tolist(); hyps = preds["pred"].fillna("").astype(str).tolist()
    cer, wer = cer_wer([norm_basic(r) for r in refs], [norm_basic(h) for h in hyps])
    cer_n, wer_n = cer_wer([norm_strict(r) for r in refs], [norm_strict(h) for h in hyps])
    lat = preds["latency_s"].to_numpy()
    return {"n": len(preds), "CER (%)": round(cer, 2), "WER (%)": round(wer, 2),
            "CER norm (%)": round(cer_n, 2), "WER norm (%)": round(wer_n, 2),
            "Latencia media (s)": round(float(lat.mean()), 3),
            "Latencia p95 (s)": round(float(np.percentile(lat, 95)), 3),
            "% alucinación": round(100 * np.mean([len(h) > 2 * len(r) + 10 for r, h in zip(refs, hyps)]), 2),
            "% vacías": round(100 * np.mean([len(h.strip()) == 0 for h in hyps]), 2),
            "Errores": int((preds["error"].fillna("") != "").sum())}

def free_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache(); torch.cuda.ipc_collect()

def gpu_report(tag=""):
    for i in range(torch.cuda.device_count()):
        print(f"  [{tag}] GPU{i}: asignada {torch.cuda.memory_allocated(i)/1e9:.2f} GB")

def reset_peaks():
    for i in range(torch.cuda.device_count()): torch.cuda.reset_peak_memory_stats(i)

def peak_vram_gb():
    return sum(torch.cuda.max_memory_allocated(i) for i in range(torch.cuda.device_count())) / 1e9

In [ ]:
# 3. Datos Rodrigo: misma muestra fija de 500 líneas (seed 42)
def stem(p): return os.path.splitext(os.path.basename(str(p).strip()))[0]

def parse_transcription(t):
    t = t.strip()
    return t.replace(" ", "").replace("<space>", " ") if "<space>" in t else t

def load_rodrigo():
    imgs = {stem(p): p for p in glob.glob(f"{RODRIGO}/images/**/*", recursive=True)
            if p.lower().endswith((".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp"))}
    texts = {}
    with open(f"{RODRIGO}/text/transcriptions.txt", encoding="utf-8", errors="replace") as f:
        for line in f:
            parts = line.rstrip("\n").split(maxsplit=1)
            if len(parts) == 2: texts[stem(parts[0])] = parse_transcription(parts[1])
    wanted = {stem(l) for l in open(f"{RODRIGO}/partitions/test.txt") if l.strip()}
    ids = [i for i in sorted(set(imgs) & set(texts)) if i in wanted]
    d = pd.DataFrame({"id": ids, "image_path": [imgs[i] for i in ids], "text": [texts[i] for i in ids]})
    return d[d.text.str.strip().str.len() > 0].reset_index(drop=True)

DATA = {}
if RODRIGO:
    df_all = load_rodrigo()
    DATA["rodrigo"] = df_all.sample(n=min(N_SAMPLES, len(df_all)), random_state=SEED).reset_index(drop=True)
    DATA["rodrigo"].to_csv(OUT["rodrigo"] / "muestra_test.csv", index=False)
    print(f"Rodrigo test: {len(df_all)} líneas → muestra {len(DATA['rodrigo'])} "
          f"({DATA['rodrigo'].text.str.len().sum()} caracteres)")
else:
    print("⚠️ No se encontró rodri-dataset: se omiten las corridas de Rodrigo")

In [ ]:
#!find /kaggle/input -maxdepth 4 -name "synth.zip"

In [ ]:
# 4. Datos sintéticos: MISMO split que el entrenamiento (C3: seed 42, 60/20/20 por historia)
if SYNTH_ZIP:
    dst = WORK / "synth_src"
    if not list(dst.glob("**/lines.jsonl")):
        with zipfile.ZipFile(SYNTH_ZIP) as z: z.extractall(dst)
    SRC = Path(sorted(dst.glob("**/lines.jsonl"))[0]).parent
    rows = [json.loads(l) for l in open(SRC / "lines.jsonl", encoding="utf-8") if l.strip()]
    for r in rows:
        r["image"] = str(SRC / "lines" / Path(r["image"]).name)
    pages = sorted({r["id"].rsplit("_", 1)[0] for r in rows})
    random.Random(42).shuffle(pages)
    n = len(pages)
    grupo = {p: "train" for p in pages[:int(.6 * n)]}
    grupo.update({p: "val" for p in pages[int(.6 * n):int(.8 * n)]})
    grupo.update({p: "test" for p in pages[int(.8 * n):]})
    conteo = {s: sum(grupo[r["id"].rsplit("_", 1)[0]] == s for r in rows) for s in ("train", "val", "test")}
    for s in ("train", "val", "test"):
        print(f"{s}: {conteo[s]} líneas ({sum(v == s for v in grupo.values())} historias)")
    if conteo != {"train": 9255, "val": 3057, "test": 3077}:
        print("⚠️ OJO: los conteos NO coinciden con el entrenamiento (9255/3057/3077). "
              "¿Es el mismo synth.zip?")
    test = [r for r in rows if grupo[r["id"].rsplit("_", 1)[0]] == "test"]
    d = pd.DataFrame({"id": [r["id"] for r in test], "image_path": [r["image"] for r in test],
                      "text": [r["text"] for r in test]})
    d = d[d.text.str.strip().str.len() > 0]
    d = d[d.image_path.map(os.path.exists)].reset_index(drop=True)
    DATA["synth"] = d
    d.to_csv(OUT["synth"] / "muestra_test.csv", index=False)
    print(f"Test sintético: {len(d)} líneas con imagen")
else:
    print("⚠️ No se encontró synth.zip: se omiten las corridas sintéticas")

In [ ]:
# 5. Adaptador LoRA
LORA_DIR = None
if LORA_ZIP:
    dst = WORK / "lora"
    if not list(dst.glob("**/adapter_config.json")):
        with zipfile.ZipFile(LORA_ZIP) as z: z.extractall(dst)
    cands = sorted(str(p) for p in dst.glob("**/adapter_config.json"))
    finales = [c for c in cands if "checkpoint-" not in c]
    LORA_DIR = os.path.dirname((finales or cands)[-1])
    cfg = json.load(open(f"{LORA_DIR}/adapter_config.json"))
    print("LORA_DIR:", LORA_DIR)
    for k in ("base_model_name_or_path", "r", "lora_alpha", "lora_dropout", "target_modules", "task_type"):
        print(f"  {k}: {cfg.get(k)}")
    json.dump(cfg, open(WORK / "adapter_config_usado.json", "w"), indent=2)
    if "3B" not in str(cfg.get("base_model_name_or_path")):
        print("⚠️ El adaptador NO es del 3B: se omite el LoRA"); LORA_DIR = None
else:
    print("⚠️ No se encontró synth-lora.zip: se omiten las corridas LoRA")

In [ ]:
# 6. Wrappers de modelos (load / predict / unload)
class BaseOCR:
    key = name = hf_id = license = ""
    def unload(self):
        for a in ("model", "processor"):
            if hasattr(self, a): delattr(self, a)
        free_gpu()

class TrOCRWrapper(BaseOCR):
    key, name, hf_id, license = "trocr", "TrOCR-large-handwritten", "microsoft/trocr-large-handwritten", "MIT"
    def load(self):
        from transformers import TrOCRProcessor, VisionEncoderDecoderModel
        self.processor = TrOCRProcessor.from_pretrained(self.hf_id)
        self.model = VisionEncoderDecoderModel.from_pretrained(self.hf_id, torch_dtype=DTYPE).to("cuda:0").eval()
    @torch.inference_mode()
    def predict(self, img):
        pv = self.processor(images=img, return_tensors="pt").pixel_values.to("cuda:0", DTYPE)
        out = self.model.generate(pv, max_new_tokens=MAX_NEW_TOKENS, num_beams=1, do_sample=False)
        return self.processor.batch_decode(out, skip_special_tokens=True)[0]

class Florence2Wrapper(BaseOCR):
    key, name, hf_id, license = "florence2", "Florence-2-large", "florence-community/Florence-2-large", "MIT"
    TASK = "<OCR>"
    def load(self):
        from transformers import AutoProcessor, Florence2ForConditionalGeneration
        self.processor = AutoProcessor.from_pretrained(self.hf_id)
        self.model = Florence2ForConditionalGeneration.from_pretrained(self.hf_id, dtype=DTYPE).to("cuda:0").eval()
    @torch.inference_mode()
    def predict(self, img):
        inputs = self.processor(text=self.TASK, images=img, return_tensors="pt").to("cuda:0", DTYPE)
        ids = self.model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, num_beams=1, do_sample=False)
        raw = self.processor.batch_decode(ids, skip_special_tokens=False)[0]
        return self.processor.post_process_generation(raw, task=self.TASK, image_size=img.size).get(self.TASK, "")

class LlavaNextWrapper(BaseOCR):
    key, name, hf_id, license = "llava16", "LLaVA-1.6-Mistral-7B", "llava-hf/llava-v1.6-mistral-7b-hf", "Apache-2.0"
    def load(self):
        from transformers import LlavaNextProcessor, LlavaNextForConditionalGeneration
        self.processor = LlavaNextProcessor.from_pretrained(self.hf_id)
        self.model = LlavaNextForConditionalGeneration.from_pretrained(
            self.hf_id, torch_dtype=DTYPE, device_map="auto", low_cpu_mem_usage=True).eval()
    @torch.inference_mode()
    def predict(self, img):
        conv = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": PROMPT}]}]
        prompt = self.processor.apply_chat_template(conv, add_generation_prompt=True)
        inputs = self.processor(images=img, text=prompt, return_tensors="pt").to(self.model.device, DTYPE)
        out = self.model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
                                  pad_token_id=self.processor.tokenizer.eos_token_id)
        return self.processor.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

class Qwen25VLWrapper(BaseOCR):
    key, name, hf_id, license = "qwen25vl", "Qwen2.5-VL-7B-Instruct", "Qwen/Qwen2.5-VL-7B-Instruct", "Apache-2.0"
    device_map = "auto"                                   # 7B se reparte entre las 2 T4
    def load(self):
        from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
        self.processor = AutoProcessor.from_pretrained(self.hf_id, min_pixels=64*28*28, max_pixels=1280*28*28)
        self.model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
            self.hf_id, torch_dtype=DTYPE, device_map=self.device_map).eval()
    @torch.inference_mode()
    def predict(self, img):
        msgs = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": PROMPT}]}]
        text = self.processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        inputs = self.processor(text=[text], images=[img], return_tensors="pt").to(self.model.device)
        out = self.model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)
        return self.processor.batch_decode(out[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True)[0]

class Qwen25VL3BWrapper(Qwen25VLWrapper):
    key, name, hf_id = "qwen25vl3b", "Qwen2.5-VL-3B-Instruct", "Qwen/Qwen2.5-VL-3B-Instruct"
    license, device_map = "Qwen Research (no comercial)", "cuda:0"

class Qwen25VL3BLoRAWrapper(Qwen25VL3BWrapper):
    key, name = "qwen25vl3b_lora", "Qwen2.5-VL-3B-Instruct + LoRA (sintético)"
    def load(self):
        super().load()                                    # mismo processor y fp16 que el zero-shot
        from peft import PeftModel
        self.model = PeftModel.from_pretrained(self.model, LORA_DIR).merge_and_unload().eval()

WRAPPERS = {c.key: c for c in (TrOCRWrapper, Florence2Wrapper, LlavaNextWrapper,
                               Qwen25VLWrapper, Qwen25VL3BWrapper, Qwen25VL3BLoRAWrapper)}

In [ ]:
# 7. Runner: guarda predicciones por línea, mide latencia y VRAM, libera memoria siempre
def run_benchmark(wrapper, df, out_dir):
    pred_path, meta_path = out_dir / f"preds_{wrapper.key}.csv", out_dir / f"meta_{wrapper.key}.json"
    if pred_path.exists() and meta_path.exists():
        print(f"⏭  {wrapper.name} ({out_dir.name}): ya evaluado"); return
    print(f"\n{'='*70}\n▶ {wrapper.name} | test: {out_dir.name} | n={len(df)}\n{'='*70}")
    rows, load_s, peak = [], None, None
    free_gpu(); reset_peaks()
    baseline = sum(torch.cuda.memory_allocated(i) for i in range(torch.cuda.device_count())) / 1e9
    try:
        t0 = time.time(); wrapper.load(); load_s = time.time() - t0
        print(f"  Cargado en {load_s:.1f} s"); gpu_report("cargado")
        for p in df["image_path"].iloc[:N_WARMUP]:
            wrapper.predict(Image.open(p).convert("RGB"))
        for r in tqdm(df.itertuples(index=False), total=len(df), desc=wrapper.name, mininterval=30):
            img = Image.open(r.image_path).convert("RGB")
            torch.cuda.synchronize(); t = time.perf_counter(); raw, err = "", ""
            try:
                raw = wrapper.predict(img)
            except torch.cuda.OutOfMemoryError:
                err = "OOM"; free_gpu()
            except Exception as e:
                err = f"{type(e).__name__}: {e}"[:200]
            torch.cuda.synchronize()
            rows.append({"id": r.id, "ref": r.text, "raw": raw, "pred": clean_output(raw),
                         "latency_s": time.perf_counter() - t, "error": err})
        peak = peak_vram_gb() - baseline
    finally:
        wrapper.unload(); gpu_report("liberado")
    if not rows:
        raise RuntimeError("sin predicciones")
    pd.DataFrame(rows).to_csv(pred_path, index=False)
    json.dump({"name": wrapper.name, "hf_id": wrapper.hf_id, "license": wrapper.license, "load_s": load_s,
               "vram_peak_gb": peak, "n": len(rows), "transformers": transformers.__version__,
               "lora_dir": LORA_DIR if wrapper.key.endswith("_lora") else None},
              open(meta_path, "w"), indent=2)
    print(f"  ✅ {pred_path.name} | VRAM pico {peak:.2f} GB | CER {compute_metrics(pd.DataFrame(rows))['CER (%)']} %")

In [ ]:
# 8. Ejecutar el plan (cada corrida aislada: si una falla, sigue con la siguiente)
LOG = []
t_total = time.time()
for test, key, activo in PLAN:
    if not activo: continue
    if test not in DATA:
        LOG.append((test, key, "omitido: faltan datos")); continue
    if key.endswith("_lora") and not LORA_DIR:
        LOG.append((test, key, "omitido: falta adaptador")); continue
    w = WRAPPERS[key]()
    try:
        run_benchmark(w, DATA[test], OUT[test]); LOG.append((test, key, "ok"))
    except Exception as e:
        print(f"❌ {key} en {test} falló: {type(e).__name__}: {e}"); traceback.print_exc()
        LOG.append((test, key, f"ERROR {type(e).__name__}: {str(e)[:120]}"))
    finally:
        del w; free_gpu()
print(f"\nTiempo total: {(time.time()-t_total)/3600:.2f} h")
print(pd.DataFrame(LOG, columns=["test", "modelo", "estado"]).to_string(index=False))

In [ ]:
# 9. Tablas (una por test; NO se mezclan)
try:
    from transformers import AutoTokenizer
    _qtok = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-VL-3B-Instruct")
except Exception as e:
    _qtok = None; print("Sin tokenizer para % truncadas:", e)

def pct_truncadas(key, preds):
    if _qtok is None or not key.startswith("qwen"): return None
    n = [len(_qtok(str(r), add_special_tokens=False).input_ids) for r in preds["raw"].fillna("")]
    return round(100 * np.mean([k >= MAX_NEW_TOKENS - 1 for k in n]), 2)

def tabla(out_dir):
    filas = []
    for mf in sorted(out_dir.glob("meta_*.json")):
        meta = json.load(open(mf)); key = mf.stem[5:]
        preds = pd.read_csv(out_dir / f"preds_{key}.csv", keep_default_na=False)
        if len(preds) == 0: continue
        row = {"Modelo": meta["name"].replace(" + LoRA (sintético)", ""),
               "Adaptación": "LoRA (sintético)" if key.endswith("_lora") else "zero-shot"}
        row.update(compute_metrics(preds))
        row["% truncadas"] = pct_truncadas(key, preds)
        row["VRAM pico (GB)"] = round(meta["vram_peak_gb"], 2) if meta["vram_peak_gb"] else None
        row["Licencia"] = meta["license"]
        filas.append(row)
    if not filas: return None
    t = pd.DataFrame(filas).sort_values("CER (%)").reset_index(drop=True)
    t.to_csv(out_dir / "resumen.csv", index=False)
    return t

pd.set_option("display.width", 250); pd.set_option("display.max_columns", 30)
for test, titulo in (("rodrigo", "TABLA 1 — Test Rodrigo (500 líneas, seed 42)"),
                     ("synth", "TABLA 2 — Test sintético (split por historia, seed 42)")):
    t = tabla(OUT[test])
    print(f"\n{titulo} | fp16 | greedy | max_new_tokens={MAX_NEW_TOKENS} | T4")
    print(t.to_string() if t is not None else "  (sin resultados)")

In [ ]:
# 10. IC 95 % (bootstrap por línea, B=2000) y diferencia pareada zero-shot vs LoRA
def _err(preds, norm):
    refs = [norm(r) for r in preds["ref"].astype(str)]
    hyps = [norm(h) for h in preds["pred"].fillna("").astype(str)]
    return (np.array([Levenshtein.distance(r, h) for r, h in zip(refs, hyps)]),
            np.array([len(r) for r in refs]))

def bootstrap(out_dir, B=2000):
    rng = np.random.default_rng(SEED)
    zs = pd.read_csv(out_dir / "preds_qwen25vl3b.csv", keep_default_na=False).set_index("id")
    lo = pd.read_csv(out_dir / "preds_qwen25vl3b_lora.csv", keep_default_na=False).set_index("id").loc[zs.index]
    res = []
    for nombre, norm in (("CER", norm_basic), ("CER norm", norm_strict)):
        e0, n0 = _err(zs.reset_index(), norm); e1, n1 = _err(lo.reset_index(), norm)
        idx = rng.integers(0, len(n0), size=(B, len(n0)))
        c0 = 100 * e0[idx].sum(1) / n0[idx].sum(1); c1 = 100 * e1[idx].sum(1) / n1[idx].sum(1); d = c1 - c0
        p0, p1 = 100 * e0.sum() / n0.sum(), 100 * e1.sum() / n1.sum()
        res.append({"Métrica": nombre,
                    "zero-shot [IC95]": f"{p0:.2f} [{np.percentile(c0,2.5):.2f}, {np.percentile(c0,97.5):.2f}]",
                    "LoRA [IC95]": f"{p1:.2f} [{np.percentile(c1,2.5):.2f}, {np.percentile(c1,97.5):.2f}]",
                    "Δ LoRA−ZS [IC95]": f"{p1-p0:+.2f} [{np.percentile(d,2.5):+.2f}, {np.percentile(d,97.5):+.2f}]",
                    "p(Δ≥0)": round(float(np.mean(d >= 0)), 4)})
    return pd.DataFrame(res)

for test in ("rodrigo", "synth"):
    if (OUT[test] / "preds_qwen25vl3b.csv").exists() and (OUT[test] / "preds_qwen25vl3b_lora.csv").exists():
        try:
            r = bootstrap(OUT[test]); r.to_csv(OUT[test] / "bootstrap_zs_vs_lora.csv", index=False)
            print(f"\n=== {test}: Qwen 3B zero-shot vs LoRA ===\n{r.to_string(index=False)}")
        except Exception as e:
            print(f"Bootstrap {test} falló: {e}")

In [ ]:
# 11. Ejemplos cualitativos (5 líneas por test) + diagnóstico de Florence-2
for test in ("rodrigo", "synth"):
    if test not in DATA: continue
    ids = DATA[test]["id"].iloc[:5].tolist(); ej = []
    for f in sorted(OUT[test].glob("preds_*.csv")):
        p = pd.read_csv(f, keep_default_na=False)
        for _, r in p[p["id"].isin(ids)].iterrows():
            ej.append({"id": r["id"], "modelo": f.stem[6:], "referencia": r["ref"][:80], "predicción": r["pred"][:80]})
    if ej:
        e = pd.DataFrame(ej).sort_values(["id", "modelo"]); e.to_csv(OUT[test] / "ejemplos.csv", index=False)
        print(f"\n=== Ejemplos {test} ===\n{e.to_string(index=False)}")

f2 = OUT["rodrigo"] / "preds_florence2.csv"
if f2.exists():
    p = pd.read_csv(f2, keep_default_na=False)
    print("\n=== Florence-2: salida cruda vs referencia (¿error de parseo o del modelo?) ===")
    for _, r in p.head(8).iterrows():
        print(f"REF: {r['ref'][:70]!r}\nRAW: {str(r['raw'])[:70]!r}\n")

In [ ]:
# 12. Empaquetar resultados
with zipfile.ZipFile(WORK / "resultados_completos.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for d in OUT.values():
        for f in d.glob("*"): z.write(f, f"{d.name}/{f.name}")
    if (WORK / "adapter_config_usado.json").exists():
        z.write(WORK / "adapter_config_usado.json", "adapter_config_usado.json")
print("Listo →", WORK / "resultados_completos.zip", f"({(WORK/'resultados_completos.zip').stat().st_size/1e6:.1f} MB)")